In [1]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.7.1+cu118
CUDA Available: True
GPU Name: NVIDIA GeForce RTX 3060


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

Using device: cuda


In [3]:
MODEL_NAME = "VietAI/vit5-base"

MAX_SOURCE_LENGTH = 512
MAX_TARGET_LENGTH = 128

DATASET_NAME = "ithieund/VietNews-Abs-Sum"
SOURCE_COLUMN = "article"
TARGET_COLUMN = "abstract"

SEED = 42
TRAIN_SAMPLE_SIZE = 50_000
EVAL_SAMPLE_SIZE = 2_000
TEST_SAMPLE_SIZE = 2_000

CHECKPOINT_DIR = "../models/vit5-summarization"
LOG_DIR = "./logs"

# Prepare data

In [4]:
from datasets import DatasetDict, load_dataset

dataset = load_dataset(DATASET_NAME)
dataset

d:\GithubRepositories\TextSummarization\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['guid', 'title', 'abstract', 'article'],
        num_rows: 303686
    })
    validation: Dataset({
        features: ['guid', 'title', 'abstract', 'article'],
        num_rows: 67010
    })
    test: Dataset({
        features: ['guid', 'title', 'abstract', 'article'],
        num_rows: 67640
    })
})

In [5]:
sample = dataset["train"][0]
print("TITLE:\n", sample.get("title", ""))
print("\nARTICLE:\n", sample[SOURCE_COLUMN][:1200])
print("\nABSTRACT:\n", sample[TARGET_COLUMN])

TITLE:
 Khởi_tố kẻ_trộm hơn 1 tạ thóc và hơn 8 triệu đồng của chú ruột để lấy tiền mua ma_tuý

ARTICLE:
 Ngày 27/3 , Cơ_quan Cảnh_sát điều_tra Công_an TP. Hưng_Yên , tỉnh Hưng_Yên cho biết , đơn_vị vừa ra quyết_định khởi_tố vụ án , khởi_tố bị_can đối_với đối_tượng Mai_Văn_Thương ( SN 1989 , trú tại đội 11 , thôn An_Chiểu 1 , xã Liên_Phương , TP. Hưng_Yên ) để điều_tra về hành_vi trộm_cắp tài_sản . Theo tài_liệu điều_tra của cơ_quan công_an , vào_khoảng 7h30 ngày 13/3 , lợi_dụng gia_đình ông Mai_Văn_Thịnh ( chú ruột đối_tượng Thương ) ở cạnh nhà đi vắng , đối_tượng này đã đạp gãy chấn_song cửa_sổ , đột_nhập vào nhà ông Thịnh trộm_cắp 121kg thóc mang bán cho người cùng thôn lấy 700.000 đ . Không dừng lại , sau đó đối_tượng tiếp_tục quay lại lục_soát tủ nhà ông Thịnh trộm_cắp 8.500.000 đ tiền_mặt ( ông Thịnh để dưới đáy tủ ) , rồi dùng số tiền trên để đi mua ma_tuý về sử_dụng và tiêu_xài hết 6.080.000 đ . Đến ngày 15/3 , đối_tượng Thương đã đến Cơ_quan điều_tra Công_an TP. Hưng_Yên tự_thú

# Load model

In [6]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

In [7]:
def preprocess_batch(examples):
    model_inputs = tokenizer(
        examples[SOURCE_COLUMN],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        examples[TARGET_COLUMN],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [8]:
tokenized_dataset = DatasetDict({
    "train": dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SAMPLE_SIZE)).map(
        preprocess_batch, batched=True, remove_columns=dataset["train"].column_names
    ),
    "validation": dataset["validation"].shuffle(seed=SEED).select(range(EVAL_SAMPLE_SIZE)).map(
        preprocess_batch, batched=True, remove_columns=dataset["train"].column_names
    ),
    "test": dataset["test"].shuffle(seed=SEED).select(range(TEST_SAMPLE_SIZE)).map(
        preprocess_batch, batched=True, remove_columns=dataset["train"].column_names
    ),
})

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# Training args

In [10]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    
    gradient_accumulation_steps=8,

    num_train_epochs=3,
    weight_decay=0.01,

    predict_with_generate=False,
    generation_max_length=128,
    generation_num_beams=4,

    logging_dir=LOG_DIR,
    logging_steps=100,

    save_total_limit=2,

    fp16=True
)

# Evaluate Rouge score

In [11]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

# Training

In [12]:
from transformers import Seq2SeqTrainer
model.gradient_checkpointing_enable()
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    # compute_metrics=compute_metrics
)

trainer.train()

C:\Users\khanh\AppData\Local\Temp\ipykernel_7188\3498324673.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,1.609600,1.516866


KeyboardInterrupt: 

In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

# Test

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_path = "./vit5-summarization-final"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def summarize(text):
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

In [ ]:
text = """
Nội dung văn bản dài cần tóm tắt...
"""

print(summarize(text))